In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'data/prices_round_1_day_-1.csv',
    'data/prices_round_1_day_-2.csv',
    'data/prices_round_1_day_0.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [ ]:

def calculate_book_vwap(df):
    nominal = 0
    volume = 0
    for i in range(1, 4):
        nominal += (df[f'bid_price_{i}'] * df[f'bid_volume_{i}']).fillna(0)
        nominal += (df[f'ask_price_{i}'] * df[f'ask_volume_{i}']).fillna(0)
        volume += df[f'bid_volume_{i}'].fillna(0)
        volume += df[f'ask_volume_{i}'].fillna(0)
    return nominal / volume

def calculate_micro_price(df):
    """
    Calculates the volume-weighted micro-price.
    Uses Level 1 Bid/Ask prices and volumes.
    """
    bid_p, bid_v = df['bid_price_1'], df['bid_vol_1']
    ask_p, ask_v = df['ask_price_1'], df['ask_vol_1']
    
    # Standard Micro-price formula: (Pb * Va + Pa * Vb) / (Vb + Va)
    # This weights the price toward the side with more volume (liquidity)
    micro_price = (bid_p * ask_v + ask_p * bid_v) / (bid_v + ask_v)
    
    # Fallback: if volume is 0 on both sides, use Mid Price
    micro_price = micro_price.fillna((bid_p + ask_p) / 2)
    
    return micro_price

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

# Configuration
days = [-1, -2, 0]
window_size = 17 # Ticks for rolling average/median

for day in days:
    # Filter and copy subset
    subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # 1. Calculate Micro Price and Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    subset['micro_price'] = calculate_micro_price(subset)
    
    # 2. Calculate Rolling Stats
    subset['mid_price_roll_median'] = subset['mid_price'].rolling(window=window_size).median()
    subset['micro_roll_median'] = subset['micro_price'].rolling(window=window_size).median()
    
    # 3. Statistical Analysis Suite (Residuals)
    # Using Mid Price vs Median as the primary noise baseline
    subset['mid_residuals'] = subset['mid_price'] - subset['mid_price_roll_median']
    res_clean = subset['mid_residuals'].dropna()

    if not res_clean.empty:
        # Ljung-Box for Autocorrelation (White Noise Test)
        lb_results = acorr_ljungbox(res_clean, lags=[10], return_df=True)
        lb_pvalue = lb_results['lb_pvalue'].iloc[0]
        
        # Distribution Metrics
        kurt = stats.kurtosis(res_clean)
        skew = stats.skew(res_clean)
        _, norm_p = stats.normaltest(res_clean)

        print(f"\n--- Statistical Suite: Day {day} ---")
        print(f"Ljung-Box (Lag 10) p-value: {lb_pvalue:.5f}")
        print(f"  > Status: {'White Noise' if lb_pvalue > 0.05 else 'Signal Leakage (Autocorrelated)'}")
        print(f"Excess Kurtosis: {kurt:.2f}")
        print(f"  > Distribution: {'Fat-Tailed (Outlier Heavy)' if kurt > 0 else 'Thin-Tailed'}")
        print(f"Normality p-value: {norm_p:.5f}")
        print("-" * 35)

        # Optional: Distribution Plot
        fig_hist = go.Figure(data=[go.Histogram(x=res_clean, nbinsx=50, marker_color='gray')])
        fig_hist.update_layout(title=f"Residual Distribution - Day {day}", template='plotly_white')
        fig_hist.show()

    # 4. Create the Interactive Plotly Figure
    fig = go.Figure()


    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['mid_price_roll_median'], 
                             name='Mid Rolling Median', line=dict(color='darkblue', dash='dot')))
    
    # Order Book Levels
    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['bid_price_1'], 
                             name='Bid 1', line=dict(color='green', dash='dot', width=1)))
    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['ask_price_1'], 
                             name='Ask 1', line=dict(color='red', dash='dot', width=1)))


    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['micro_roll_median'], 
                             name='micro_price Rolling Median', line=dict(color='firebrick', dash='dot')))

    # Layout Customization
    fig.update_layout(
        title=f'Market Analysis: ASH_COATED_OSMIUM - Day {day}',
        xaxis_title='Timestamp',
        yaxis_title='Price',
        legend_title='Metrics',
        template='plotly_white',
        hovermode='x unified'
    )
    
    fig.show()


--- Statistical Suite: Day -1 ---
Ljung-Box (Lag 10) p-value: 0.00049
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 5.62
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------



--- Statistical Suite: Day -2 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 5.41
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------



--- Statistical Suite: Day 0 ---
Ljung-Box (Lag 10) p-value: 0.00002
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 5.61
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------


In [49]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox

def calculate_micro_price(df):
    res = (df['bid_price_1'] * df['ask_volume_1'] + df['ask_price_1'] * df['bid_volume_1']) / \
          (df['bid_volume_1'] + df['ask_volume_1'])
    return res.fillna(df['mid_price'])

# Test odd window sizes
window_candidates = range(3, 33, 2)
days = [-2, -1, 0]
all_results = []

print(f"{'W':<5} | {'Day':<5} | {'Mid p-val':<12} | {'Micro p-val':<12}")
print("-" * 45)

for W in window_candidates:
    for day in days:
        subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
        if subset.empty: continue
            
        subset['micro_price'] = calculate_micro_price(subset)
        
        # Calculate residuals
        res_mid = (subset['mid_price'] - subset['mid_price'].rolling(W).median()).dropna()
        res_mic = (subset['micro_price'] - subset['micro_price'].rolling(W).median()).dropna()
        
        # Statistical check
        p_mid = acorr_ljungbox(res_mid, lags=[10])['lb_pvalue'].iloc[0]
        p_mic = acorr_ljungbox(res_mic, lags=[10])['lb_pvalue'].iloc[0]
        
        all_results.append({'W': W, 'day': day, 'p_mid': p_mid, 'p_mic': p_mic})
        print(f"{W:<5} | {day:<5} | {p_mid:<12.5f} | {p_mic:<12.5f}")

# --- Cross-Day Analysis ---
df_res = pd.DataFrame(all_results)

# We want the SMALLEST window where the MINIMUM p-value across all days is > 0.01 (or 0.05)
# This ensures it passes the test even on the "worst" day.
summary = df_res.groupby('W').agg({'p_mid': 'min', 'p_mic': 'min'})

opt_mid_global = summary[summary['p_mid'] > 0.05].index.min()
opt_mic_global = summary[summary['p_mic'] > 0.05].index.min()

print("\n" + "="*40)
print(f"Global Optimal (Mid):   {opt_mid_global if not pd.isna(opt_mid_global) else 'Search Higher'}")
print(f"Global Optimal (Micro): {opt_mic_global if not pd.isna(opt_mic_global) else 'Search Higher'}")
print("="*40)

W     | Day   | Mid p-val    | Micro p-val 
---------------------------------------------


3     | -2    | 1.00000      | 1.00000     
3     | -1    | 1.00000      | 1.00000     
3     | 0     | 1.00000      | 1.00000     
5     | -2    | 1.00000      | 1.00000     
5     | -1    | 1.00000      | 1.00000     
5     | 0     | 1.00000      | 1.00000     
7     | -2    | 1.00000      | 1.00000     
7     | -1    | 1.00000      | 1.00000     
7     | 0     | 1.00000      | 1.00000     
9     | -2    | 1.00000      | 1.00000     
9     | -1    | 1.00000      | 1.00000     
9     | 0     | 1.00000      | 1.00000     
11    | -2    | 1.00000      | 1.00000     
11    | -1    | 1.00000      | 1.00000     
11    | 0     | 1.00000      | 1.00000     
13    | -2    | 1.00000      | 1.00000     
13    | -1    | 1.00000      | 1.00000     
13    | 0     | 1.00000      | 1.00000     
15    | -2    | 1.00000      | 1.00000     
15    | -1    | 1.00000      | 1.00000     
15    | 0     | 1.00000      | 1.00000     
17    | -2    | 1.00000      | 1.00000     
17    | -1    | 1.00000      | 1